# Interactive Scatter Plot of World Population Data
In this example, we visualize Gapminder demographic data using an interactive scatter plot. Interaction is achieved with CustonJS of the Bokeh package. We write JavaScript code to update the graphic. Interactive Bokeh plots based on JavaScript callback functions run on vscode.

In [1]:
import numpy as np
import pandas as pd
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis256, Spectral11, Turbo256, bokeh
from bokeh.models import ColumnDataSource, CustomJS, Slider, HoverTool, Label, ColorBar, Text
from bokeh.plotting import figure, show
from bokeh.layouts import column
from bokeh.io import reset_output, output_notebook

reset_output()
output_notebook()

# Load the data
lex_female = pd.read_csv("../../data/gapminder_life_expectancy_female.csv")
lex_male = pd.read_csv("../../data/gapminder_life_expectancy_male.csv")
lex = pd.read_csv("../../data/gapminder_life_expectancy_total.csv")
gdp_pcap = pd.read_csv("../../data/gapminder_gdp_pcap.csv")
gdp_pcap.head()

# clear data 
lex_female.dropna(inplace=True)
lex_male.dropna(inplace=True)
lex.dropna(inplace=True)
gdp_pcap.dropna(inplace=True)
# like tables of the same size 
gdp_pcap = gdp_pcap.drop(gdp_pcap.index.difference(lex.index))
lex = lex.drop(lex.index.difference(gdp_pcap.index))

# gdp_pcap contains mixed data types, get numeric values
def get_numeric(value):
    """
    Convert a value to a numeric type, if possible.
    """
    if type(value) == float or type(value) == int:
        return value
    if type(value) == str and 'k' in value:
        return float(value.replace('k', '')) * 1000
    if type(value) == str:
        return float(value)

# clean data 
def clean_data(d):
    """
    Clean the data by removing non-numeric values and converting to numeric.
    """
    nr_rows = len(d)
    nr_cols = len(d[0])
    # iterator only over given columns
    for row in range(0, nr_rows):
        for col in range(0, nr_cols):
            d[row][col] = get_numeric(d[row][col])
    


# cofigure
min_year = '1800'
max_year = '2024'
min_lex = lex.loc[:,str(min_year):str(max_year)].min().min()
max_lex = lex.loc[:,str(min_year):str(max_year)].max().max()

# prepare data for plotting
countries = gdp_pcap['country'].values.tolist()
# drop the first column (country names) and convert to numpy array
gdp_pcap = gdp_pcap.drop(columns=['country']).to_numpy()
lex = lex.drop(columns=['country']).to_numpy()
lex = lex.T
gdp = gdp_pcap.T
gda = clean_data(gdp)

min_gdp = np.array(gdp).min()
max_gdp = np.array(gdp).max()

# use CustomJS to update the plot, pass data in config dict
config = {
    'min_rad': 10,
    'max_rad': 50,
    'min_year': min_year,
    'max_year': max_year,
    'gdp': gdp.tolist(),
    'lex': lex.tolist(),
    'countries': countries,
}

# compute the size of the circles
def radius(gpd, row, config):
    """
    Compute the radius of the circles based on GDP per capita
    """
    min_rad = config['min_rad']
    max_rad = config['max_rad']
    gdp = config['gdp']
    min_gdp = gpd[row].min()
    max_gdp = gpd[row].max()
    nr_cols = len(gdp[row])
    # scale the GDP per capita to a range of 0 to 1
    radii = []
    for col in range(0, nr_cols):
        # scale the GDP per capita to a range of 0 to 1
        scaled_gdp = (gpd[row][col] - min_gdp) / (max_gdp - min_gdp)
        # scale the radius to a range of min_rad to max_rad
        radii.append(min_rad + (scaled_gdp * (max_rad - min_rad)))
    return radii


# create data source
source = ColumnDataSource(data=dict(
    x=gdp[0],
    y=lex[0],
    radius=radius(gdp, 0, config),
    country=countries,
))

# colors
mapper = linear_cmap(field_name='x', palette='Viridis256', low=gdp[0].min(), high=gdp[0].max())


# figure
title = 'Life Expectancy vs GDP per Capita' #+ ' (' + min_year + ')'
width = 800
height = 600
p = figure(title='Life Expectancy vs GDP per Capita',width=width, height=height, x_axis_type='log')
p.xaxis.axis_label = 'GDP per Capita'
p.yaxis.axis_label = 'Life Expectancy'
#p.x_range.start = gdp.min()
#p.x_range.end = gdp.max()
p.y_range.start = lex.min()
p.y_range.end = lex.max()


# text 
label = Label(x=10, y=10, x_units='screen', y_units='screen', text='Year ' + min_year, text_color="#96deb3", text_font_size ='64px')
p.add_layout(label)

# scatter plot
p.scatter(
    x='x', 
    y='y', 
    source=source, 
    size='radius', 
    fill_color=mapper, 
    line_color='black', 
    alpha=0.6,
    #legend_label='Year ' + min_year,
    name='gdp_scatter',)
# tooltip
tooltips = [
    ('Country', '@country'),
    ('GDP per capita (millions)', '@x{0,0}'),
    ('Life expectancy', '@y{0.00}'),
]
p.add_tools(HoverTool(
    tooltips=tooltips,
    renderers=[p.renderers[0]],
    mode='mouse',
    point_policy='follow_mouse',))

# CustomJS callback
slider = Slider(start=1800, end=2024, value=1800, step=1, title='Year') 
callback = CustomJS(args=dict(
    source=source, 
    config=config, 
    mapper=mapper,
    label=label,
    p=p), 
    code="""
    const year = cb_obj.value
    const row = year - 1800
    const gdp = config.gdp[row]
    const lex = config.lex[row]
    const countries = config.countries
    const min_rad = config.min_rad
    const max_rad = config.max_rad
    const min_gdp = Math.min(...gdp)
    const max_gdp = Math.max(...gdp)
    const min_lex = Math.min(...lex)
    const max_lex = Math.min(...lex)
    const radii = []
    
    for (let i = 0; i < gdp.length; i++) {
        const scaled_gdp = (gdp[i] - min_gdp) / (max_gdp - min_gdp)
        const radius = min_rad + (scaled_gdp * (max_rad - min_rad))
        radii.push(radius)
    }
    source.data = {
        x: gdp,
        y: lex,
        radius: radii,
        country: countries,
    }
    mapper.high = max_gdp
    mapper.low = min_gdp
    label.text = 'Year ' + year
    """)

slider.js_on_change('value', callback)

layout = column([slider, p])
show(layout)



Loading BokehJS ...